In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [3]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
이전 예제에서는 `messages` 상태 키에 기본 `operator.add` 또는 `+` 리듀서를 적용했습니다.
이 방식은 항상 새 메시지를 기존 메시지 배열 끝에 추가합니다.

이제 기존 메시지를 교체하는 기능을 지원하기 위해,
같은 `id`를 가진 메시지는 교체하고 그렇지 않으면 추가하는
커스텀 리듀서 함수로 `messages` 키를 정의합니다.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # ID가 없는 메시지에 ID 할당
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # 새 메시지를 기존 메시지와 병합
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # 같은 ID를 가진 기존 메시지는 교체
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # 새 메시지는 끝에 추가
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [4]:
_tavily = TavilySearch(max_results=2)

@tool
def tavily_search(query: str) -> str:
    """Search the web for current information about a topic."""
    return str(_tavily.invoke({"query": query}))

tool_instance = tavily_search

## Manual Human Approval

In [5]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]  # action 노드 실행 전 인터럽트
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_ollama(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            args = {"query": t['args']['query']}  # query 인자만 전달
            result = self.tools[t['name']].invoke(args)
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [6]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOllama(model="llama3.1:8b")
abot = Agent(model, [tool_instance], system=prompt, checkpointer=memory)

In [7]:
messages = [HumanMessage(content="Whats the weather in Seoul?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in Seoul?', additional_kwargs={}, response_metadata={}, id='d17e1826-a955-4aa6-9a77-173965ab3c8a'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:08:50.426014Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6609932750, 'load_duration': 5545944750, 'prompt_eval_count': 221, 'prompt_eval_duration': 560906917, 'eval_count': 21, 'eval_duration': 442432040, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaaff-611f-7f60-9f77-1bfe47686387-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Seoul weather'}, 'id': 'da47469d-1eeb-423a-b5d2-015dc2fd5b8e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 21, 'total_tokens': 242})]}
{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:08:5

In [8]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Seoul?', additional_kwargs={}, response_metadata={}, id='d17e1826-a955-4aa6-9a77-173965ab3c8a'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:08:50.426014Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6609932750, 'load_duration': 5545944750, 'prompt_eval_count': 221, 'prompt_eval_duration': 560906917, 'eval_count': 21, 'eval_duration': 442432040, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaaff-611f-7f60-9f77-1bfe47686387-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Seoul weather'}, 'id': 'da47469d-1eeb-423a-b5d2-015dc2fd5b8e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 21, 'total_tokens': 242})]}, next=('action',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f163c9a-f35

In [9]:
abot.graph.get_state(thread).next

('action',)

### continue after interrupt

In [10]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search', 'args': {'query': 'Seoul weather'}, 'id': 'da47469d-1eeb-423a-b5d2-015dc2fd5b8e', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'Seoul weather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Seoul\', \'url\': \'https://www.weatherapi.com/\', \'content\': "{\'location\': {\'name\': \'Seoul\', \'region\': \'\', \'country\': \'South Korea\', \'lat\': 37.5664, \'lon\': 126.9997, \'tz_id\': \'Asia/Seoul\', \'localtime_epoch\': 1780985484, \'localtime\': \'2026-06-09 15:11\'}, \'current\': {\'last_updated_epoch\': 1780984800, \'last_updated\': \'2026-06-09 15:00\', \'temp_c\': 25.2, \'temp_f\': 77.4, \'is_day\': 1, \'condition\': {\'text\': \'Sunny\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/day/113.png\', \'code\': 1000}, \'wind_mph\': 11.6, \'wind_kph\': 18.7, \'wind_degree\': 268, \'wind_dir\': \'W\', \'pressure_mb\': 1007.0, \'pressure_in\': 29.74,

In [11]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in Seoul?', additional_kwargs={}, response_metadata={}, id='d17e1826-a955-4aa6-9a77-173965ab3c8a'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:08:50.426014Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6609932750, 'load_duration': 5545944750, 'prompt_eval_count': 221, 'prompt_eval_duration': 560906917, 'eval_count': 21, 'eval_duration': 442432040, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaaff-611f-7f60-9f77-1bfe47686387-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Seoul weather'}, 'id': 'da47469d-1eeb-423a-b5d2-015dc2fd5b8e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 21, 'total_tokens': 242}), ToolMessage(content='{\'query\': \'Seoul weather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [

In [12]:
abot.graph.get_state(thread).next

()

In [13]:
# 1단계: 실행 → action 노드 직전에서 인터럽트
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

print("\n--- 인터럽트 발생. 아래 셀을 실행하면 계속 진행됩니다. ---")
print("현재 대기 중인 노드:", abot.graph.get_state(thread).next)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='035337c9-dd55-4d5d-a1dd-65aee15bd692'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:12:55.764872Z', 'done': True, 'done_reason': 'stop', 'total_duration': 897259375, 'load_duration': 103588666, 'prompt_eval_count': 221, 'prompt_eval_duration': 359850709, 'eval_count': 20, 'eval_duration': 420006832, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eab03-35d0-7183-bba3-34945ef341bd-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'LA weather'}, 'id': 'f0e9d2ca-04f1-4411-96fd-96b437fc0be6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 20, 'total_tokens': 241})]}
{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T06:12:55.764872

In [14]:
# 2단계: 이 셀을 실행하는 것 자체가 승인 (Human Approval)
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search', 'args': {'query': 'LA weather'}, 'id': 'f0e9d2ca-04f1-4411-96fd-96b437fc0be6', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'LA weather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Los Angeles, California\', \'url\': \'https://www.weatherapi.com/\', \'content\': "{\'location\': {\'name\': \'Los Angeles\', \'region\': \'California\', \'country\': \'United States of America\', \'lat\': 34.0522, \'lon\': -118.2428, \'tz_id\': \'America/Los_Angeles\', \'localtime_epoch\': 1780985600, \'localtime\': \'2026-06-08 23:13\'}, \'current\': {\'last_updated_epoch\': 1780984800, \'last_updated\': \'2026-06-08 23:00\', \'temp_c\': 17.8, \'temp_f\': 64.0, \'is_day\': 0, \'condition\': {\'text\': \'Clear\', \'icon\': \'//cdn.weatherapi.com/weather/64x64/night/113.png\', \'code\': 1000}, \'wind_mph\': 2.9, \'wind_kph\': 4.7, \'wind_degree\': 207, \'wind_dir\': \'SS

## 상태 수정 (Modify State)
인터럽트까지 실행한 후 상태를 수정합니다.

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
current_values = abot.graph.get_state(thread)

In [ ]:
current_values.values['messages'][-1]

In [ ]:
current_values.values['messages'][-1].tool_calls

In [ ]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search',
     'args': {'query': 'current weather in Louisiana'},
     'id': _id}
]

In [ ]:
abot.graph.update_state(thread, current_values.values)

In [ ]:
abot.graph.get_state(thread)

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

## Time Travel

In [ ]:
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

In [ ]:
to_replay = states[-3]

In [ ]:
to_replay

In [ ]:
for event in abot.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)

## Go back in time and edit

In [ ]:
to_replay

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [ ]:
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

In [ ]:
for event in abot.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

## Add message to a state at a given time

In [ ]:
to_replay

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']

In [ ]:
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name="tavily_search",
    content="54 degree celcius",
)]}

In [ ]:
branch_and_add = abot.graph.update_state(
    to_replay.config, 
    state_update, 
    as_node="action")

In [ ]:
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)

# Extra Practice

## 간단한 그래프 만들기 (Build a small graph)
상태 메모리 제어에 대한 이해를 높이고 싶다면 직접 다뤄볼 수 있는 간단한 그래프입니다.

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.memory import MemorySaver

다음 상태를 가진 간단한 2-노드 그래프를 정의합니다:
- `lnode`: 마지막으로 실행된 노드
- `scratch`: 임시 메모 공간
- `count`: 각 단계마다 증가하는 카운터

In [ ]:
class AgentState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]

In [ ]:
def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

그래프는 N1→N2→N1 순서로 실행되며, count가 3에 도달하면 종료됩니다.

In [ ]:
def should_continue(state):
    return state["count"] < 3

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", 
                              should_continue, 
                              {True: "Node1", False: END})
builder.set_entry_point("Node1")

In [ ]:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

### 실행해보기 (Run it!)
thread를 설정하고 실행합니다!

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
graph.invoke({"count":0, "scratch":"hi"},thread)

### 현재 상태 확인 (Look at current state)

현재 상태를 가져옵니다. AgentState인 `values`를 확인하세요. 아래에서 스냅샷을 참조할 때 사용할 `config`와 `thread_ts`도 확인하세요.

In [ ]:
graph.get_state(thread)

메모리에 있는 모든 상태 스냅샷을 확인합니다. AgentState 변수인 `count`를 통해 내용을 추적할 수 있습니다. 가장 최근 스냅샷이 먼저 반환됩니다. 메타데이터의 `step` 변수는 그래프 실행 단계 수를 나타냅니다. 또한 `parent_config`는 이전 노드의 `config`임을 확인할 수 있습니다.

### 상태 이력 확인 (Look at state history)

In [ ]:
for state in graph.get_state_history(thread):
    print(state, "\n")

`config`만 리스트에 저장합니다. 오른쪽의 count 순서를 확인하세요. `get_state_history`는 가장 최근 스냅샷을 먼저 반환합니다.

In [ ]:
states = []
for state in graph.get_state_history(thread):
    states.append(state.config)
    print(state.config, state.values['count'])

이전 상태를 가져옵니다.

In [ ]:
states[-3]

이것은 Node1이 처음 완료된 후의 상태입니다. `next`가 `Node2`이고 `count`가 1임을 확인하세요.

In [ ]:
graph.get_state(states[-3])

### 시간 되돌리기 (Go Back in Time)
`invoke`에서 해당 상태를 사용하여 시간을 되돌립니다. states[-3]을 현재 상태로 사용하여 node2부터 계속 실행되는 것을 확인하세요.

In [ ]:
graph.invoke(None, states[-3])

새로운 상태들이 상태 이력에 추가된 것을 확인하세요. 오른쪽의 count 값을 확인해보세요.

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state.config, state.values['count'])

아래에서 자세한 내용을 확인할 수 있습니다. 텍스트가 많지만 새로운 브랜치가 시작되는 노드를 찾아보세요. parent `config`가 스택의 이전 항목이 아니라 state[-3]의 항목임을 확인하세요.

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state,"\n")

### 상태 수정 (Modify State)
새로운 thread로 시작하여 이력을 초기화합니다.

In [ ]:
thread2 = {"configurable": {"thread_id": str(2)}}
graph.invoke({"count":0, "scratch":"hi"},thread2)

In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_png())

In [ ]:
states2 = []
for state in graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values['count'])   

상태 하나를 가져옵니다.

In [ ]:
save_state = graph.get_state(states2[-3])
save_state

이제 값을 수정합니다. 한 가지 주의할 점: AgentState 정의 시 `count`는 `operator.add`를 사용했습니다. 즉, 값이 현재 count에 *더해집니다*. 여기서 `-3`은 현재 count 값에서 차감됩니다.

In [ ]:
save_state.values["count"] = -3
save_state.values["scratch"] = "hello"
save_state

이제 상태를 업데이트합니다. 이렇게 하면 메모리의 맨 위, 즉 최신 항목으로 새로운 항목이 생성됩니다. 이것이 현재 상태가 됩니다.

In [ ]:
graph.update_state(thread2,save_state.values)

현재 상태가 맨 위에 있습니다. `thread_ts`를 통해 확인할 수 있습니다.
새 노드의 `parent_config`와 `thread_ts`를 확인하세요 - 이전 노드의 것입니다.

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

### `as_node`로 다시 시도하기 (Try again with `as_node`)
`update_state()`로 상태를 작성할 때, 어떤 노드가 작성자라고 가정할지 그래프 로직에 알려주어야 합니다. 이렇게 하면 그래프 로직이 그래프에서 해당 노드를 찾을 수 있습니다. 값을 작성한 후, `next()` 값은 새 상태를 사용하여 그래프를 순회함으로써 계산됩니다. 이 경우 `Node1`이 상태를 작성했으므로, 그래프는 다음 상태를 `Node2`로 계산할 수 있습니다. 조건부 엣지를 통과하는 경우도 있으니 주의하세요!

In [ ]:
graph.update_state(thread2,save_state.values, as_node="Node1")

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

특정 `thread_ts`를 지정하지 않으면 `invoke`는 현재 상태(방금 추가된 항목)에서 실행됩니다.

In [ ]:
graph.invoke(None,thread2)

상태 이력을 출력합니다. 최신 항목에서 `scratch` 값이 변경된 것을 확인하세요.

In [ ]:
for state in graph.get_state_history(thread2):
    print(state,"\n")

계속 실험해보세요!